In [3]:
"""
Construction d’un masque terre à partir de données d’élévation (DEM).

Ce script :
- télécharge (ou utilise localement) des tuiles DEM via dem_stitcher,
- assemble les tuiles sur une zone d’intérêt,
- crée un masque binaire terre/mer (terre = 1, mer = 0),
- sauvegarde le masque au format GeoTIFF.

Dépendances :
- dem_stitcher
- earthpy
- rasterio
- numpy

!!! Attention :
Sur un nœud de calcul sans accès Internet, il est nécessaire de :
1) télécharger les tuiles DEM en amont,
2) définir explicitement le chemin vers ces données locales (variable `data_dir`).
"""

import os
import numpy as np
import rasterio as rio
import earthpy as et
from dem_stitcher import stitch_dem

# ============================================================
# Paramètres généraux
# ============================================================

# Région d’étude (utilisée uniquement pour le nom de sortie)
region = "senegal2"

# Choix du jeu de données DEM
# Valeurs possibles :
# ['glo_30', 'srtm_v3', 'glo_90', 'nasadem', '3dep', 'glo_90_missing']
choose_data = "glo_30"

# Zone d’intérêt (bounding box) : [lon_min, lat_min, lon_max, lat_max]
bounds = [-19, 12, -16, 17]  # Sénégal

# ============================================================
# Gestion des données DEM
# ============================================================

# Si accès Internet :
# Télécharge automatiquement les données nécessaires
data = et.data.get_data("vignette-elevation")

# Si PAS d’accès Internet, commenter la ligne ci-dessus et utiliser :
# data_dir = "/chemin/vers/les/tuiles/dem/telechargees/"
output_path = (
    "/espace/home/datawork-cersat-public/cache/users/trishna/"
    "processing_KC/landmask/dem/"
    f"landmask_copDEM_{choose_data}_{region}.tif"
)

# ============================================================
# Assemblage des tuiles DEM
# ============================================================

elevation, metadata = stitch_dem(
    bounds,
    dem_name=choose_data,
    dst_ellipsoidal_height=False,
    dst_area_or_point="Point",
    # data_dir=data_dir,  # à décommenter si données locales
    # download=False      # utile sur cluster sans Internet
)

# ============================================================
# Création du masque terre / mer
# ============================================================

# Copie de la donnée d’élévation
landmask = elevation.copy()

# ------------------------------------------------------------
# Nettoyage du masque (AUCUN NaN autorisé)
# ------------------------------------------------------------
landmask = np.where(landmask > 0, 1, 0).astype("uint8")

# ------------------------------------------------------------
# Métadonnées de sortie
# ------------------------------------------------------------
out_meta = metadata.copy()

#  SUPPRIMER toute trace de nodata NaN
out_meta.pop("nodata", None)

#  Forcer des métadonnées cohérentes
out_meta.update({
    "dtype": "uint8",
    "count": 1,
    "nodata": 0
})

# ------------------------------------------------------------
# Écriture du GeoTIFF
# ------------------------------------------------------------
with rio.open(output_path, "w", **out_meta) as dst:
    dst.write(landmask, 1)

print(f"Masque terre sauvegardé : {output_path}")


Reading tile imagery: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:05<00:00,  1.17it/s]


Masque terre sauvegardé : /espace/home/datawork-cersat-public/cache/users/trishna/processing_KC/landmask/dem/landmask_copDEM_glo_30_senegal2.tif


# Pour utilisation dans un scripte:
Voici une fonction réutilisable, pensée pour fonctionner aussi bien en local que sur cluster.
Elle encapsule tout : DEM → masque terre → GeoTIFF en sortie.

Exemple d’utilisation

In [14]:
def build_landmask_from_dem(
    bounds,
    output_path,
    dem_name="glo_30",
    region_name="region"
):
    """
    Construction d’un masque terre/mer à partir de données d’élévation (DEM).

    Le masque est défini comme :
        - terre : 1 (élévation > 0)
        - mer   : 0 (élévation <= 0)

    Paramètres
    ----------
    bounds : list or tuple
        Bounding box [lon_min, lat_min, lon_max, lat_max].
    output_path : str
        Chemin du fichier GeoTIFF de sortie.
    dem_name : str, optionnel
        Jeu de données DEM utilisé.
    region_name : str, optionnel
        Nom de la région (informatif, non utilisé dans le calcul).

    Retours
    -------
    landmask : numpy.ndarray
        Masque terre/mer (uint8).
    metadata : dict
        Métadonnées raster associées.
    """

    # ------------------------------------------------------------
    # Assemblage des tuiles DEM
    # ------------------------------------------------------------
    elevation, metadata = stitch_dem(
        bounds,
        dem_name=dem_name,
        dst_ellipsoidal_height=False,
        dst_area_or_point="Point"
    )

    # ------------------------------------------------------------
    # Création du masque terre / mer
    # ------------------------------------------------------------
    landmask = np.where(elevation > 0, 1, 0).astype("uint8")

    # ------------------------------------------------------------
    # Métadonnées de sortie
    # ------------------------------------------------------------
    out_meta = metadata.copy()
    out_meta.pop("nodata", None)
    out_meta.update({
        "dtype": "uint8",
        "count": 1,
        "nodata": 0
    })

    # ------------------------------------------------------------
    # Écriture du GeoTIFF
    # ------------------------------------------------------------
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with rio.open(output_path, "w", **out_meta) as dst:
        dst.write(landmask, 1)

    print(f"Masque terre sauvegardé : {output_path}")

    return landmask, out_meta


In [15]:
bounds = [-19, 12, -16, 17]  # Sénégal

output_file = (
    "/espace/home/datawork-cersat-public/cache/users/trishna/"
    "processing_KC/landmask/dem/landmask_copDEM_glo_30_senegal2f.tif"
)

landmask, metadata = build_landmask_from_dem(
    bounds=bounds,
    output_path=output_file,
    dem_name="glo_30",
    region_name="senegal",
)


Reading tile imagery: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:05<00:00,  1.17it/s]


Masque terre sauvegardé : /espace/home/datawork-cersat-public/cache/users/trishna/processing_KC/landmask/dem/landmask_copDEM_glo_30_senegal2f.tif
